# Notebook 03 — Modèle BM25+

**Milestone M-03 | Étudiant B — IR Specialist**

Ce notebook couvre :
1. Indexation BM25+ avec la librairie `rank_bm25`
2. Fonction de recherche `search(query, k)`
3. Évaluation : Recall@10, Precision@10, MRR
4. Comparaison qualitative BM25+ vs TF-IDF

## 1. Imports

In [ ]:
import json
import os
import pickle

import numpy as np
import pandas as pd
from rank_bm25 import BM25Plus

DATA_DIR    = '../data'
OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('Imports OK')

## 2. Chargement des Données

In [ ]:
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

tokenized_corpus = [doc.split() for doc in df_docs['content_clean']]

def get_relevant_ids(qgts, query_id):
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]

def build_query_text(q):
    parts = [q.get('text', ''), q.get('title', '')]
    if q.get('tags'):
        parts.append(' '.join(q['tags']))
    return ' '.join(p for p in parts if p).strip()

print(f'Documents chargés      : {len(tokenized_corpus):,}')
print(f'Requêtes train         : {len(queries_train)}')
print(f'Requêtes test          : {len(queries_test)}')
print(f'Exemple tokens (doc 0) : {tokenized_corpus[0][:8]}...')

## 3. Indexation BM25+

In [ ]:
# BM25+ : amélioration de BM25 classique
# Paramètres par défaut : k1=1.5, b=0.75, delta=0.5
# delta > 0 garantit un score non nul pour tous les termes qui apparaissent
bm25 = BM25Plus(tokenized_corpus)

print(f'Index BM25+ construit.')
print(f'Taille du vocabulaire : {len(bm25.idf)}')
print(f'Longueur moyenne des documents : {bm25.avgdl:.1f} tokens')

## 4. Fonction de Recherche

In [ ]:
def search(query: str, k: int = 10) -> dict:
    """
    Recherche BM25+ sur le corpus.

    Args:
        query: Requête en langage naturel.
        k: Nombre de résultats à retourner.

    Returns:
        dict avec 'topk_indices' et 'topk_scores'.
    """
    query_tokens = query.lower().split()
    scores       = bm25.get_scores(query_tokens)
    top_k_idx    = np.argsort(scores)[::-1][:k]
    return {
        'topk_indices': top_k_idx.tolist(),
        'topk_scores':  scores[top_k_idx].tolist()
    }

# Test
result = search('neural networks deep learning', k=5)
print('Top 5 résultats BM25+ pour "neural networks deep learning" :')
for idx, score in zip(result['topk_indices'], result['topk_scores']):
    print(f"  [{score:.4f}] {df_docs.iloc[idx]['id']} — {df_docs.iloc[idx]['title']}")

## 5. Métriques d'Évaluation

In [ ]:
def recall_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    if not relevant_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(relevant_set)


def precision_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    return len(retrieved_set & relevant_set) / k


def mrr(retrieved_indices, relevant_ids, id_to_idx):
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant_set:
            return 1.0 / rank
    return 0.0

print('Métriques définies.')

## 6. Évaluation BM25+

In [ ]:
K = 10
recalls, precisions, mrrs_list = [], [], []

for query_entry in queries_train:
    query_id     = query_entry['id']
    query_text   = build_query_text(query_entry)
    relevant_ids = get_relevant_ids(qgts_train, query_id)

    if not relevant_ids:
        continue

    result    = search(query_text, k=K)
    retrieved = result['topk_indices']

    recalls.append(recall_at_k(retrieved, relevant_ids, K, id_to_idx))
    precisions.append(precision_at_k(retrieved, relevant_ids, K, id_to_idx))
    mrrs_list.append(mrr(retrieved, relevant_ids, id_to_idx))

bm25_results = {
    'model':           'BM25+',
    f'Recall@{K}':     np.mean(recalls),
    f'Precision@{K}':  np.mean(precisions),
    'MRR':             np.mean(mrrs_list)
}

print(f'Évaluation sur {len(recalls)} requêtes (avec ground truth)')
print('=== Résultats BM25+ ===')
for key, val in bm25_results.items():
    if isinstance(val, float):
        print(f'{key:15s}: {val:.4f}')
    else:
        print(f'{key:15s}: {val}')

## 7. Comparaison BM25+ vs TF-IDF

In [ ]:
# Chargement des résultats TF-IDF
with open(os.path.join(OUTPUTS_DIR, 'tfidf_results.pkl'), 'rb') as f:
    tfidf_results = pickle.load(f)

comparison = pd.DataFrame([tfidf_results, bm25_results]).set_index('model')
print('=== Tableau Comparatif TF-IDF vs BM25+ ===')
print(comparison.to_string())

diff = comparison.loc['BM25+'] - comparison.loc['TF-IDF']
print()
print('Delta (BM25+ - TF-IDF) :')
print(diff.to_string())

### 7.1 Analyse Qualitative : Cas où BM25+ Surpasse TF-IDF

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
with open('../models/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)

def search_tfidf(query, k=10):
    vec    = vectorizer.transform([query])
    scores = cos_sim(vec, tfidf_matrix).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return {'topk_indices': top_k.tolist(), 'topk_scores': scores[top_k].tolist()}

# 3 requêtes représentatives
sample_queries = [queries_train[0], queries_train[10], queries_train[20]]

for q_entry in sample_queries:
    q_text   = build_query_text(q_entry)
    q_id     = q_entry['id']
    relevant = get_relevant_ids(qgts_train, q_id)
    if not relevant:
        continue

    res_tf  = search_tfidf(q_text, k=5)
    res_bm  = search(q_text, k=5)

    mrr_tf = mrr(res_tf['topk_indices'], relevant, id_to_idx)
    mrr_bm = mrr(res_bm['topk_indices'], relevant, id_to_idx)

    print(f'Requête : "{q_text[:70]}"')
    print(f'  TF-IDF MRR={mrr_tf:.3f} | BM25+ MRR={mrr_bm:.3f}  → BM25+ {"MEILLEUR" if mrr_bm > mrr_tf else "ÉGAL/INFÉRIEUR"}')
    print('  Top-3 BM25+ :')
    for idx, score in zip(res_bm['topk_indices'][:3], res_bm['topk_scores'][:3]):
        doc_id = idx_to_id[idx]
        marker = '✓' if doc_id in relevant else ' '
        print(f"    {marker} [{score:.3f}] {doc_id} — {df_docs.iloc[idx]['title'][:55]}")
    print()

In [ ]:
# Sauvegarde des résultats BM25+
with open(os.path.join(OUTPUTS_DIR, 'bm25_results.pkl'), 'wb') as f:
    pickle.dump(bm25_results, f)

# Sauvegarde de l'index BM25+ pour réutilisation (re-ranking Phase 2)
with open('../models/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

print('Résultats et index BM25+ sauvegardés.')

## 8. Résumé

Le modèle BM25+ :
- Utilise `BM25Plus` de `rank_bm25` (paramètres : k1=1.5, b=0.75, delta=0.5)
- Attribue un score non nul à tous les termes présents dans la requête
- Gère mieux la normalisation par longueur de document que TF-IDF
- Interface : `search(query, k) -> {'topk_indices': [...], 'topk_scores': [...]}`

**Avantages de BM25+ par rapport à TF-IDF :**
- Normalisation par longueur de document plus fine (paramètre `b`)
- Saturation du TF avec `k1` — les mots très fréquents sont moins dominants
- Paramètre `delta` évite les scores nuls pour les termes présents